# Process Pattern Anomaly Detection - Seq2Seq Autoencoder + Isolation Forest

## Overview

This notebook uses a hybrid approach combining Deep Learning (Seq2Seq Autoencoder) with statistical methods (Isolation Forest) to detect complex process behavior patterns. The solution analyzes Sysmon logs to identify anomalous process sequences.

**Objective:** Detect complex process behavior patterns that deviate from normal baseline using sequence reconstruction and statistical outlier detection.

**Data Requirements:**
- Sysmon Operational logs
- Essential Event IDs: 1 (Process Create), 3 (Network), 7 (Image Loaded), 11 (File Create), 22 (DNS), 17/18 (Pipes) for context
- ProcessGuid field for sessionization
- Data readable from MinIO/S3

**Expected Outputs:**
- Trained Seq2Seq autoencoder model
- Trained Isolation Forest model
- Anomaly scores for all process instances
- Combined ensemble score (LSTM + Isolation Forest)
- Top anomalous processes ranked by final score
- Confusion matrix and classification metrics

**Model Approach:**
**Seq2Seq Autoencoder:**
- Tokenize process event sequences (event IDs)
- Train sequence model (next-token prediction)
- Calculate reconstruction error as anomaly score

**Isolation Forest:**
- Extract numerical features (event counts, timing, rarity)
- Train unsupervised model to identify statistical outliers
- Binary classification (anomaly vs normal)

**Ensemble Fusion:**
- Combine LSTM anomaly score + Isolation Forest prediction
- Apply Z-score normalization per process image
- Apply rarity boost for unique/singleton processes
- Weighted sum for final anomaly score

**Prerequisites:**
- MinIO/S3 access configured in environment variables
- Spark cluster with GPU access
- Sysmon logs available
- PyTorch with GPU support

**Notes:**
- Developed in test environment; tuning is needed for your dataset
- Model ensemble is highly sensitive to process frequency in your network
- Thresholds require adjustment based on false positive rate
- Can be adapted for other data sources (EDR solutions)

In [ ]:
# Cell 1: Imports & Environment Setup
# =========================================

import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import ArrayType, LongType
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pyspark.ml.torch.distributor import TorchDistributor
import s3fs
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as _sum, desc
from pyspark.sql.types import FloatType, IntegerType
from sklearn.ensemble import IsolationForest
import traceback
load_dotenv()

print("✅ Environment loaded")

In [ ]:
# Cell 2: Spark Configuration
# =========================================

SPARK_APP_NAME = "Process_Anomaly_Detection"

SPARK_MASTER = os.getenv('SPARK_MASTER')
SPARK_DRIVER_HOST = os.getenv('SPARK_DRIVER_HOST')
SPARK_DRIVER_PORT = int(os.getenv('SPARK_DRIVER_PORT', '7771'))
SPARK_BLOCK_MANAGER_PORT = int(os.getenv('SPARK_BLOCK_MANAGER_PORT', '7772'))
SPARK_UI_PORT = int(os.getenv('SPARK_UI_PORT', '8088'))
SPARK_EXECUTOR_CORES = int(os.getenv('SPARK_EXECUTOR_CORES', '16'))
SPARK_EXECUTOR_MEMORY = os.getenv('SPARK_EXECUTOR_MEMORY', '32G')
SPARK_DRIVER_MEMORY = os.getenv('SPARK_DRIVER_MEMORY', '2g')
SPARK_EXECUTOR_GPU_AMOUNT = os.getenv('SPARK_EXECUTOR_GPU_AMOUNT', '1')
SPARK_TASK_GPU_AMOUNT = os.getenv('SPARK_TASK_GPU_AMOUNT', '1')
SPARK_JARS_PATH = os.getenv('SPARK_JARS_PATH', '/home/jmi/Spark_cluster/master/jars/*')
SPARK_EXECUTOR_CLASSPATH = os.getenv('SPARK_EXECUTOR_CLASSPATH', '/opt/spark/jars/*:/opt/spark/external-jars/*')
USE_GPU = os.getenv('USE_GPU', 'false').lower() == 'true'

print(f"✅ Spark Config: master={SPARK_MASTER}, cores={SPARK_EXECUTOR_CORES}, memory={SPARK_EXECUTOR_MEMORY}, gpu={USE_GPU}")

In [ ]:
# Cell 3: MinIO Configuration
# =========================================

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
MINIO_PATH_STYLE_ACCESS = os.getenv('MINIO_PATH_STYLE_ACCESS', 'true')

STORAGE_OPTIONS = {
    'key': MINIO_ACCESS_KEY,
    'secret': MINIO_SECRET_KEY,
    'endpoint_url': MINIO_ENDPOINT
}

print(f"✅ MinIO Config: endpoint={MINIO_ENDPOINT}")

In [ ]:
# Cell 4: Data Paths
# =========================================

DATA_PATH = os.getenv('DATA_PATH', 's3a://winlogbeat/winlogbeat/*.parquet')
BASE_S3_PATH = os.getenv('BASE_S3_PATH', 's3a://temp/anomaly_detection')

print(f"✅ Data Path: {DATA_PATH}")

In [ ]:
# Cell 5: Process Anomaly Parameters
# =========================================

PROCESS_MAX_SEQ_LEN = int(os.getenv('PROCESS_MAX_SEQ_LEN', '128'))
PROCESS_VOCAB_SIZE = int(os.getenv('PROCESS_VOCAB_SIZE', '30'))
PROCESS_EMBEDDING_DIM = int(os.getenv('PROCESS_EMBEDDING_DIM', '64'))
PROCESS_HIDDEN_DIM = int(os.getenv('PROCESS_HIDDEN_DIM', '256'))
PROCESS_LATENT_DIM = int(os.getenv('PROCESS_LATENT_DIM', '64'))
PROCESS_NUM_LAYERS = int(os.getenv('PROCESS_NUM_LAYERS', '2'))
PROCESS_NUM_EPOCHS = int(os.getenv('PROCESS_NUM_EPOCHS', '10'))
PROCESS_IF_N_ESTIMATORS = int(os.getenv('PROCESS_IF_N_ESTIMATORS', '100'))
PROCESS_IF_CONTAMINATION = float(os.getenv('PROCESS_IF_CONTAMINATION', '0.05'))
PROCESS_IF_RANDOM_STATE = int(os.getenv('PROCESS_IF_RANDOM_STATE', '42'))
PROCESS_FINAL_ANOMALY_THRESHOLD = float(os.getenv('PROCESS_FINAL_ANOMALY_THRESHOLD', '3.3000890575199424'))
PROCESS_AUTOENCODER_BATCH_SIZE = int(os.getenv('PROCESS_AUTOENCODER_BATCH_SIZE', '64'))

print(f"✅ Process Anomaly Config: seq_len={PROCESS_MAX_SEQ_LEN}, vocab={PROCESS_VOCAB_SIZE}, if_estimators={PROCESS_IF_N_ESTIMATORS}")

In [ ]:
# Cell 6: Create Spark Session
# =========================================

builder = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)

    .config("spark.driver.host", SPARK_DRIVER_HOST)
    .config("spark.driver.port", str(SPARK_DRIVER_PORT))
    .config("spark.blockManager.port", str(SPARK_BLOCK_MANAGER_PORT))
    .config("spark.ui.port", str(SPARK_UI_PORT))

    .config("spark.executor.cores", str(SPARK_EXECUTOR_CORES))
    .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)

    .config("spark.executor.extraClassPath", SPARK_EXECUTOR_CLASSPATH)
    .config("spark.jars", SPARK_JARS_PATH)
    .config("spark.executor.userClassPathFirst", "true")

    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", MINIO_PATH_STYLE_ACCESS)
)

if USE_GPU:
    builder = (
        builder
        .config("spark.executor.resource.gpu.amount", SPARK_EXECUTOR_GPU_AMOUNT)
        .config("spark.task.resource.gpu.amount", SPARK_TASK_GPU_AMOUNT)
    )

spark = builder.getOrCreate()

print(f"✅ Spark session created: {SPARK_APP_NAME}")
print(f"   Master: {SPARK_MASTER}")
print(f"   GPU: {USE_GPU}")

In [ ]:
# Cell 7: Load Data
# =========================================

df_raw = spark.read.parquet(DATA_PATH)
print(f"✅ Loaded {df_raw.count()} records from MinIO")

## Data Filtering

The following code:
1. Adds a human-readable time column from epoch time
2. Filters data based on needs (Sysmon Operational, non-null ProcessGuid, specific event IDs)
3. Removes unnecessary columns to reduce memory footprint

**Key Fields:**
- `event_created`: Event creation time in epoch
- `winlog_event_id`: Sysmon event ID
- `winlog_event_data_processguid`: Unique process session identifier
- `winlog_computer_name`: Source hostname

In [ ]:
# Cell 8: Filter Data
# =========================================

df_with_time = df_raw.withColumn(
    "readable_time",
    F.from_unixtime(F.col("`event_created`") / 1000000000)
)

df_filtered = (
    df_with_time
    .where(F.col("winlog_channel") == "Microsoft-Windows-Sysmon/Operational")
    .where(F.col("winlog_event_id").isin([1, 3, 7, 11, 22, 17, 18, 19, 20, 21]))
    .where(F.col("winlog_event_data_processguid").isNotNull())
    .drop(
        "_index", "_id", "_timestamp", "@timestamp", "@version", 
        "tags", "message", "ecs_version", "event_kind", "event_type", 
        "event_module", "event_dataset", "agent_ephemeral_id", "agent_id", 
        "agent_name", "agent_type", "agent_version",
        "winlog_keywords", "winlog_activity_id", "winlog_provider_guid",
        "winlog_event_data_utctime", "winlog_event_data_creationutctime", 
        "winlog_user_identifier", "log_level", "event_action",
        "winlog_event_data_configurationfilehash","winlog_event_data_configuration",
        "winlog_event_data_elevatedtoken","winlog_event_data_workstationname",
        "winlog_event_data_ipport","winlog_event_data_targetdomainname",
        "winlog_event_data_authenticationpackagename","winlog_event_data_targetusername",
        "winlog_event_data_subjectdomainname","winlog_event_data_targetusersid",
        "winlog_event_data_subjectusersid","winlog_event_data_restrictedadminmode",
        "winlog_event_data_impersonationlevel","winlog_event_data_targetlogonid",
        "winlog_event_data_transmittedservices","winlog_event_data_subjectlogonid",
        "winlog_event_data_virtualaccount","winlog_event_data_subjectusername",
        "winlog_event_data_targetoutboundusername", "winlog_event_data_scriptblockid",
        "winlog_event_data_targetoutbounddomainname","winlog_event_data_lmpackagename",
        "winlog_event_data_targetlinkedlogonid","winlog_event_data_service",
        "winlog_event_data_objectserver","winlog_event_data_privilegelist",
        "winlog_event_data_accessmask","winlog_event_data_objecttype",
        "winlog_event_data_objectname","winlog_event_data_accesslist",
        "winlog_event_data_calltrace",
        "winlog_event_data_resourceattributes","winlog_event_data_keylength",
        "winlog_event_data_messagenumber","winlog_event_data_messagetotal",
        "winlog_event_data_schemaversion","winlog_event_data_version"
    )
)

print(f"✅ Filtered to {df_filtered.count()} records (Sysmon Ops, event IDs [1,3,7,11,22,17,18,19,20,21], non-null processguid)")

df_filtered.printSchema()
df_filtered.show(5, truncate=False)

## Sessionization

Converts the "vertical" stream of individual log events into "horizontal" feature sets per process session (ProcessGuid).

**Process:**
1. Group all events by `winlog_event_data_processguid`
2. Collect all event data fields into lists
3. Calculate session metadata (start/end time, event counts)

**Result:** Each ProcessGuid becomes a feature vector representing that process's complete lifecycle

**Note:** This is a compute-heavy operation designed for GPU acceleration.

In [ ]:
# Cell 9: Sessionize Data
# =========================================

print("Starting Sessionization...")

df_filtered.persist(StorageLevel.MEMORY_AND_DISK)

event_data_cols = [c for c in df_filtered.columns if c.startswith("winlog_event_data_") and c != "winlog_event_data_processguid"]

agg_exprs = [F.collect_list(c).alias(c) for c in event_data_cols]

agg_exprs.extend([
    F.count("*").alias("total_events"),
    F.min("`event_created`").alias("start_time"),
    F.max("`event_created`").alias("end_time"),
    F.collect_list("winlog_event_id").alias("event_id_sequence")
])

df_sessionized = (
    df_filtered
    .groupBy("winlog_event_data_processguid")
    .agg(*agg_exprs)
)

df_sessionized = df_sessionized.withColumn(
    "duration_seconds",
    (F.col("end_time") - F.col("start_time")) / 1000000000
)

df_sessionized = df_sessionized.withColumn(
    "event_id_sequence",
    F.sort_array("event_id_sequence")
)

print(f"✅ Sessionization complete. {df_sessionized.count()} sessions created")

## Seq2Seq Model Training

This section implements a Seq2Seq Autoencoder for sequence anomaly detection:

**Architecture:**
- Input: Tokenized event ID sequences
- Encoder: LSTM (or similar) layers
- Decoder: Reconstructs input sequence
- Training: Next-token prediction (x: [t1, t2, t3] → y: [t2, t3, 0])
- Loss: Cross-entropy between predicted and actual token
- Inference: Reconstruction error (high error = anomalous)

**Hyperparameters:**
- `MAX_SEQ_LEN`: Maximum sequence length (128)
- `VOCAB_SIZE`: Number of unique event IDs (30)
- `EMBEDDING_DIM`: Token embedding dimension (64)
- `HIDDEN_DIM`: LSTM hidden dimension (256)
- `LATENT_DIM`: Latent space dimension (64)
- `NUM_LAYERS`: Number of LSTM layers (2)
- `NUM_EPOCHS`: Training epochs (10)
- `BATCH_SIZE`: Batch size (64)

**Distributed Training:**
- Use TorchDistributor for GPU acceleration
- Workers read training data from MinIO/S3
- Each worker loads PyTorch model on GPU
- Reduces training time significantly

In [ ]:
# Cell 10: Prepare Seq2Seq Training Data
# =========================================

TRAIN_DATA_PATH = f"{BASE_S3_PATH}/seq2seq_train.parquet"
MODEL_S3_PATH = f"{BASE_S3_PATH}/seq2seq_autoenc.pth"

def prepare_sequences(event_ids_series):
    x_data = []
    y_data = []
    for seq in event_ids_series:
        seq = seq[:PROCESS_MAX_SEQ_LEN]
        arr = np.array(seq, dtype=np.int64)
        x_pad = np.pad(arr, (0, PROCESS_MAX_SEQ_LEN - len(arr)), mode='constant')
        x_data.append(x_pad.tolist())
        y_data.append(x_pad.tolist()) # Target is same as input
    return pd.DataFrame({'x': x_data, 'y': y_data})

prepare_udf = F.pandas_udf(prepare_sequences, "x:array<long>, y:array<long>")

df_prepared = (
    df_sessionized
    .select("winlog_event_data_processguid", "event_id_sequence")
    .filter(F.size("event_id_sequence") > 2)
    .withColumn("processed", prepare_udf("event_id_sequence"))
    .select("winlog_event_data_processguid", F.col("processed.x").alias("x"), F.col("processed.y").alias("y"))
)


df_prepared.repartition(100).write.mode("overwrite").parquet(TRAIN_DATA_PATH)

print(f"✅ Training data saved to {TRAIN_DATA_PATH}")

In [ ]:
# Cell 11: Seq2Seq Training Function
# =========================================

# Autoencoder LSTM
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        # We return the hidden and cell states as the "Latent Vector"
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell):
        # x is usually the previous output (during inference), but for training 
        # we can feed the true input (Teacher Forcing) to help convergence
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        predictions = self.fc(outputs)
        return predictions, hidden, cell

class Seq2SeqAutoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source):
        # Encode the source sequence
        hidden, cell = self.encoder(source)
        
        # Decode to reconstruct
        # In an Autoencoder, we feed the input as the initial target (Teacher Forcing)
        outputs, _, _ = self.decoder(source, hidden, cell)
        return outputs

def train_seq2seq(s3_path_for_pandas, storage_options):
    """
    Train Seq2Seq Autoencoder on GPU using TorchDistributor.

    Args:
        s3_path_for_pandas: Path to train data parquet file
        storage_options: S3 connection options

    Returns:
        None (saves model to MinIO)
    """
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[WORKER] Training on device: {device}")

        df = pd.read_parquet(s3_path_for_pandas, storage_options=storage_options)

        if df.empty:
            raise ValueError("[WORKER] Dataframe is empty! Check S3 path.")

        x = np.array(df['x'].tolist(), dtype=np.int64)
        y = np.array(df['y'].tolist(), dtype=np.int64)

        train_data = torch.utils.data.TensorDataset(torch.tensor(x), torch.tensor(y))
        train_loader = DataLoader(train_data, batch_size=PROCESS_AUTOENCODER_BATCH_SIZE, shuffle=True)

        enc = Encoder(PROCESS_VOCAB_SIZE, PROCESS_EMBEDDING_DIM, PROCESS_HIDDEN_DIM, PROCESS_NUM_LAYERS)
        dec = Decoder(PROCESS_VOCAB_SIZE, PROCESS_EMBEDDING_DIM, PROCESS_HIDDEN_DIM, PROCESS_NUM_LAYERS)
        model = Seq2SeqAutoencoder(enc, dec).to(device)
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        model.train()
        total_loss = 0
        for epoch in range(PROCESS_NUM_EPOCHS):
            for i, (inp, tgt) in enumerate(train_loader):
                inp, tgt = inp.to(device), tgt.to(device)

                optimizer.zero_grad()
                prediction = model(inp)
                batch_size, seq_len, vocab_size = prediction.shape
                prediction = prediction.reshape(batch_size * seq_len, vocab_size)
                loss = criterion(prediction.reshape(-1, PROCESS_VOCAB_SIZE), tgt.reshape(-1))
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            print(f"[WORKER] Epoch {epoch+1}/{PROCESS_NUM_EPOCHS}, Loss: {total_loss/len(train_loader):.6f}")

        model_s3_path = MODEL_S3_PATH
        fs = s3fs.S3FileSystem(**storage_options)
        with fs.open(model_s3_path, "wb") as f:
            torch.save(model.state_dict(), f)
        print(f"[WORKER] Model saved to {model_s3_path}")

    except Exception as e:
        print("[WORKER] ERROR:")
        traceback.print_exc()
        raise e

print("✅ Training function defined")

In [ ]:
# Cell 12: Launch Distributed Training
# =========================================

print("Starting Phase: Distributed Seq2Seq Training on GPU...")

distributor = TorchDistributor(
    num_processes=1,
    local_mode=False,
    use_gpu=USE_GPU
)

distributor.run(train_seq2seq, TRAIN_DATA_PATH, STORAGE_OPTIONS)

print("✅ Phase: Distributed Training complete")

In [ ]:
# Cell 13: Inference Function
# =========================================

def score_batch_seq2seq(iterator):
    """
    Score batches using trained Seq2Seq Autoencoder.

    Args:
        iterator: Pandas DataFrame iterator

    Returns:
        Yields: DataFrame with anomaly_score added
    """
    fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
    with fs.open(MODEL_S3_PATH, "rb") as f:
        # Load architecture
        enc = Encoder(PROCESS_VOCAB_SIZE, PROCESS_EMBEDDING_DIM, PROCESS_HIDDEN_DIM, PROCESS_NUM_LAYERS)
        dec = Decoder(PROCESS_VOCAB_SIZE, PROCESS_EMBEDDING_DIM, PROCESS_HIDDEN_DIM, PROCESS_NUM_LAYERS)
        model = Seq2SeqAutoencoder(enc, dec)
        model.load_state_dict(torch.load(f))
        model.eval()
        model.to('cuda')

    criterion = nn.CrossEntropyLoss(reduction='none', ignore_index=0)

    for batch_pdf in iterator:
        x = torch.tensor(batch_pdf['x'].tolist(), dtype=torch.long).cuda()
        y = torch.tensor(batch_pdf['y'].tolist(), dtype=torch.long).cuda()

        with torch.no_grad():
            # Reconstruct
            outputs = model(x)
            
            # Reconstruction Error
            loss = criterion(outputs.view(-1, PROCESS_VOCAB_SIZE), y.view(-1))
            loss = loss.view(x.shape[0], x.shape[1])
            
            mask = (y != 0).float()
            valid_counts = mask.sum(dim=1).clamp(min=1)
            avg_loss = (loss * mask).sum(dim=1) / valid_counts
            
            batch_pdf['anomaly_score'] = avg_loss.cpu().numpy()
        yield batch_pdf


print("✅ Inference function defined")

In [ ]:
# Cell 14: Calculate Seq2Seq Anomaly Scores
# =========================================

print("Starting Phase: Seq2Seq Inference...")

result_schema = "winlog_event_data_processguid string, x array<long>, y array<long>, anomaly_score float"

df_seq2seq_results = df_prepared.mapInPandas(
    score_batch_seq2seq,
    schema=result_schema
)

print(f"✅ Seq2Seq inference complete. {df_seq2seq_results.count()} scores calculated")

## Feature Engineering & Rarity Calculation

Extract rarity features to boost anomaly scores for unique processes.

**Features:**
- IP frequency (how rare is destination IP?)
- Domain frequency (how rare is DNS query?)
- Image frequency (how rare is binary?)
- Parent image frequency

**Approach:**
- Group by IP, domain, image, parent image
- Calculate counts across all sessions
- Join back to sessionized data
- Calculate rarity: -log2(count) where higher is more anomalous

In [ ]:
# Cell 15: Calculate Rarity
# =========================================

def calculate_rarity(df, column_name):
    # Select ONLY the guid and the target column
    temp_df = df.select("winlog_event_data_processguid", column_name)
    
    # Repartition heavily before exploding to avoid OOM
    temp_df = temp_df.repartition(2000)
    
    # Explode and Count
    counts = (
        temp_df
        .select(F.explode(column_name).alias("val"))
        .filter(F.col("val").isNotNull())
        .groupBy("val")
        .agg(F.count("*").alias("count"))
    )
    return counts

# Calculate Frequencies Separately
ip_counts = calculate_rarity(df_sessionized, "winlog_event_data_destinationip")
domain_counts = calculate_rarity(df_sessionized, "winlog_event_data_queryname")
# Process image is usually unique per row, but cheap enough to do here
image_counts = calculate_rarity(df_sessionized, "winlog_event_data_image") 
# Parent image counts
parent_image_counts = calculate_rarity(df_sessionized, "winlog_event_data_parentimage") 

# Rename columns to 'item' and '*_freq' to ensure consistency
ip_counts = ip_counts.withColumnRenamed("val", "item").withColumnRenamed("count", "ip_freq")
domain_counts = domain_counts.withColumnRenamed("val", "item").withColumnRenamed("count", "domain_freq")
image_counts = image_counts.withColumnRenamed("val", "item").withColumnRenamed("count", "image_freq")
parent_image_counts = parent_image_counts.withColumnRenamed("val", "item").withColumnRenamed("count", "parent_image_freq")

print("✅ Rarity counts calculated")

In [ ]:
# Cell 16: Calculate Rareness Scores
# =========================================

GlobalRarity = (
    ip_counts
    .join(domain_counts, "item", "full_outer")
    .join(image_counts, "item", "full_outer")
    .join(parent_image_counts, "item", "full_outer")
    .na.fill(0, subset=["ip_freq", "domain_freq", "image_freq", "parent_image_freq"])
)
# Calculate Rareness (Join with df_sessionized)
RarenessData = (
    df_sessionized
    .select("winlog_event_data_processguid", 
            F.explode(F.col("winlog_event_data_destinationip")).alias("item"))
    .join(GlobalRarity, "item", "left")
    .select("winlog_event_data_processguid", "ip_freq")
    
    .unionByName(
        df_sessionized
        .select("winlog_event_data_processguid", 
                F.explode(F.col("winlog_event_data_queryname")).alias("item"))
        .join(GlobalRarity, "item", "left")
        .select("winlog_event_data_processguid", F.col("domain_freq").alias("ip_freq"))
    )
    .unionByName(
        df_sessionized
        .select("winlog_event_data_processguid", 
                F.when(F.size("winlog_event_data_image") > 0, F.col("winlog_event_data_image")[0]).otherwise("UNKNOWN").alias("item"))
        .join(GlobalRarity, "item", "left")
        .select("winlog_event_data_processguid", F.col("image_freq").alias("ip_freq"))
    )
    .unionByName(
        df_sessionized
        .select("winlog_event_data_processguid", 
                F.when(F.size("winlog_event_data_parentimage") > 0, F.col("winlog_event_data_parentimage")[0]).otherwise("UNKNOWN").alias("item"))
        .join(GlobalRarity, "item", "left")
        .select("winlog_event_data_processguid", F.col("parent_image_freq").alias("ip_freq"))
    )
    .groupBy("winlog_event_data_processguid")
    .agg(F.min("ip_freq").alias("rarest_freq"))
    .na.fill(999999)
)

print(f"✅ Rareness scores calculated. {RarenessData.count()} sessions enriched")

In [ ]:
# Cell 17: Build Feature Data
# =========================================

print("Starting Phase: Build Feature Data for Isolation Forest...")
FeatureData = (
    df_sessionized
    .join(RarenessData, "winlog_event_data_processguid", "left")
    .withColumn("process_image", F.when(F.size("winlog_event_data_image") > 0, F.col("winlog_event_data_image")[0]).otherwise("UNKNOWN"))
    .withColumn("cmd_line", F.when(F.size("winlog_event_data_commandline") > 0, F.col("winlog_event_data_commandline")[0]).otherwise("NONE"))
    .withColumn("parent_cmd_line", F.when(F.size("winlog_event_data_parentcommandline") > 0, F.col("winlog_event_data_parentcommandline")[0]).otherwise("NONE"))
    .withColumn("has_parent_process", F.when(F.size("winlog_event_data_parentimage") > 0, 1).otherwise(0))
    
    # --- Numerical Features ---
    .withColumn("event_count", F.size(F.col("event_id_sequence")))
    .withColumn("net_conn_count", F.size(F.col("winlog_event_data_destinationip")))
    .withColumn("dns_query_count", F.size(F.col("winlog_event_data_queryname")))
    .withColumn("file_create_count", F.size(F.col("winlog_event_data_targetfilename")))
    .withColumn("cmd_line_length", F.length(F.col("cmd_line")))
    .withColumn("parent_cmd_line_length", F.length(F.col("parent_cmd_line")))

    # Join Rareness Feature
    .withColumn("rarity_score", -F.log2(F.col("rarest_freq")))
    
    .select(
        "winlog_event_data_processguid",
        "event_count", "net_conn_count", "dns_query_count", 
        "file_create_count", "cmd_line_length", "parent_cmd_line_length", "rarity_score",
        "event_id_sequence"
    )
)

print(f"✅ Feature data built. {FeatureData.count()} sessions with features")

## Isolation Forest Training

This section trains an Isolation Forest model to detect statistical outliers in process behavior:

**Model:**
- Unsupervised anomaly detection algorithm
- Identifies outliers by isolating instances in high-dimensional space
- Binary classification: anomaly (1) vs normal (0)

**Features:**
- Event counts (network, DNS, file operations)
- Timing statistics (duration, intervals)
- Sequence features (from sessionization)

**Hyperparameters:**
- `n_estimators`: Number of trees (100)
- `contamination`: Expected anomaly ratio (0.05)
- `random_state`: Reproducibility (42)

**Approach:**
- Train on features derived from sessionized data
- Apply model to all process instances
- Get binary predictions
- Combine with Seq2Seq scores later

In [ ]:
# Cell 19: Define Isolation Forest training function
# =========================================
def fit_isolation_forest(iterator):
    MAX_SEQ_LEN = 30 

    numeric_features = [
        "event_count", "net_conn_count", "dns_query_count", 
        "file_create_count", "cmd_line_length", "parent_cmd_line_length", "rarity_score"
    ]
    
    # FIX: Iterate over the iterator provided by Spark
    for pdf in iterator:
        # Extract data safely
        guids = pdf['winlog_event_data_processguid'].tolist()
        X_numeric = pdf[numeric_features].fillna(0).to_numpy()
        
        # Convert lists of events into a dense matrix of shape (N, MAX_SEQ_LEN)
        seq_data = pdf['event_id_sequence'].tolist()
        seq_df = pd.DataFrame(seq_data)

        # Pad or Truncate
        if len(seq_df.columns) < MAX_SEQ_LEN:
            # Add padding columns (0) for missing steps
            for i in range(MAX_SEQ_LEN - len(seq_df.columns)):
                seq_df[len(seq_df.columns)] = 0
        else:
            # Truncate if sequence is too long
            seq_df = seq_df.iloc[:, :MAX_SEQ_LEN]

        X_seq = seq_df.fillna(0).to_numpy()

        # Combine: Numeric + Sequence Steps
        X_final = np.hstack([X_numeric, X_seq])
        
        # Fit Model
        clf = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
        predictions = clf.fit_predict(X_final)
        
        # Return Results
        scores = np.where(predictions == -1, 1.0, 0.0)
        yield pd.DataFrame({
            'processguid': guids,
            'if_anomaly_score': scores
        })

print(f"✅ Function defined.")

In [ ]:
# Cell 20: Train Isolation Forest
# =========================================

print("Starting Phase: Train Isolation Forest on Feature Data... may take some time")

df_if_results = FeatureData.mapInPandas(fit_isolation_forest, schema="processguid string, if_anomaly_score float")

print(f"✅ Isolation Forest training complete. {df_if_results.count()} predictions generated")

## Ensemble Scoring

Combine Seq2Seq and Isolation Forest scores into a final anomaly score.

**Scoring Logic:**
1. Normalize scores (min-max scaling)
2. Calculate Z-score per process image
3. Apply rarity boost (high for unique/singleton processes)
4. Weighted sum to produce final anomaly score

**Formula:**
final_score = (IF * 0.3) + (LSTM_norm * 0.3) + (Z_score * 1.0) + (rarity_boost * 3.0)

Tuning:** Weights can be adjusted for desired sensitivity:
- Higher IF weight: More emphasis on statistical outliers
- Higher LSTM weight: More emphasis on sequence anomalies
- Higher Z-score weight: More emphasis on process-specific deviations
- Higher rarity weight: More emphasis on unique processes

In [ ]:
# Cell 21: Calculate Final Ensemble Scores
# =========================================
print("⚠️ This will take time. With the training dataset it took around 20 minutes. ⚠️ ")
FinalEnsemble = (
    df_if_results.withColumnRenamed("processguid", "winlog_event_data_processguid")
    .join(
        df_seq2seq_results,
        "winlog_event_data_processguid",
    )
    .join(
        RarenessData,
        "winlog_event_data_processguid",
    )
    .withColumn("ensemble_score", F.col("anomaly_score") + F.col("if_anomaly_score"))
    .join(df_sessionized, "winlog_event_data_processguid")
)

ProcessBaselines = (
    FinalEnsemble
    .groupBy("winlog_event_data_image")
    .agg(
        F.mean("ensemble_score").alias("mean_score_per_image"),
        F.stddev("ensemble_score").alias("stdev_score_per_image"),
        F.count("*").alias("instance_count")
    )
)

ContextualEnsemble = (
    FinalEnsemble
    .join(ProcessBaselines, "winlog_event_data_image", "left")
    # LOGIC: If it runs only once, it is HIGHLY ANOMALOUS (Z-Score = 5.0).
    # Otherwise calculate normally. Avoid division by zero.
    .withColumn("z_score",
        F.when(F.col("stdev_score_per_image").isNull(), 5.0) 
         .when(F.col("stdev_score_per_image") == 0, 0.0)
         .otherwise((F.col("ensemble_score") - F.col("mean_score_per_image")) / F.col("stdev_score_per_image"))
    )
)

stats = (
    ContextualEnsemble
    .agg(
        F.min("ensemble_score").alias("min_ensemble"), F.max("ensemble_score").alias("max_ensemble"),
        F.min("anomaly_score").alias("min_lstm"), F.max("anomaly_score").alias("max_lstm"),
        F.min("z_score").alias("min_z"), F.max("z_score").alias("max_z")
    )
    .first()
)
# Persist results for faster access
ContextualEnsemble.persist(StorageLevel.MEMORY_AND_DISK)
ContextualEnsemble.count() 
print(f"✅ Ensemble Scores calculated.")

In [ ]:
# Cell 22: Calculate Normalized Scores & Final Score
# =========================================
lstm_norm = (F.col("anomaly_score") - stats.min_lstm) / F.greatest(F.lit(stats.max_lstm - stats.min_lstm), F.lit(0.01))

if_norm = (F.col("if_anomaly_score"))

rarity_boost = 1.0 / (1 + F.col("instance_count").cast(FloatType()) * 0.0001)

FinalResults = ContextualEnsemble.withColumn(
    "final_anomaly_score",
    (lstm_norm * 0.3) + (if_norm * 0.3) + (rarity_boost * 3.0)
)

print(f"✅ Final anomaly scores calculated. {FinalResults.count()} sessions scored")

# Display anomalies & calculate metrics

The following two cells are for analyzing the output. The following cells after can be further developed to display metrics, however, they do not represent correct results by default.

In [ ]:
# Cell 27: Display Top Anomalies
# =========================================

print("=== Top 10 Final Anomalies ===")

FinalResults.where().orderBy(F.col("final_anomaly_score").desc()).show(10, truncate=False)

print(f"Total anomalies above threshold: {FinalResults.filter(F.col('final_anomaly_score') > THRESHOLD).count()}")

In [ ]:
# Cell 28: Display more detailed results, set anomaly score limitation.
# =========================================
windowSpec = Window.partitionBy(
    "winlog_event_data_commandline",
    F.when(F.size("winlog_event_data_parentimage") > 0, F.col("winlog_event_data_parentimage")[0]).otherwise("NONE"),
    F.when(F.size("winlog_event_data_parentcommandline") > 0, F.col("winlog_event_data_parentcommandline")[0]).otherwise("NONE"),
    "winlog_event_data_image"
)

FinalResults.withColumn("parent_image", 
    F.when(F.size("winlog_event_data_parentimage") > 0, F.col("winlog_event_data_parentimage")[0]).otherwise("NONE")
).withColumn("duplicate_count", F.count("*").over(windowSpec)) \
.select(
    "winlog_event_data_processguid",
    "winlog_event_data_image",
    "winlog_event_data_commandline",
    F.when(F.size("winlog_event_data_parentimage") > 0, F.col("winlog_event_data_parentimage")[0]).otherwise("NONE").alias("parent_image"),
    F.when(F.size("winlog_event_data_parentcommandline") > 0, F.col("winlog_event_data_parentcommandline")[0]).otherwise("NONE").alias("parent_cmdline"),
    "instance_count",
    "z_score",
    "ensemble_score",
    "if_anomaly_score",
    "anomaly_score",
    "final_anomaly_score",
    "duplicate_count"
) \
 .where(F.col("final_anomaly_score") > 3.3000890575199424) \
 .orderBy(F.col("duplicate_count").desc(), F.col("final_anomaly_score").desc()) \
 .show(50, truncate=False)

In [ ]:
# Cell 23: Metrics Helper Function
# =========================================

def calculate_metrics(tp, fp, fn, tn, total_events=None):
    """
    Calculate classification metrics with standard output format.

    Args:
        tp: True Positives
        fp: False Positives
        fn: False Negatives
        tn: True Negatives
        total_events: Optional total for report

    Returns:
        dict with all metrics or None if error
    """
    try:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        print("\n" + "=" * 40)
        print("CLASSIFICATION METRICS REPORT")
        print("=" * 40)
        if total_events:
            print(f"Total Events Evaluated: {total_events}")
        print(f"True Positives (TP):  {tp:,.0f}")
        print(f"False Positives (FP): {fp:,.0f}")
        print(f"False Negatives (FN): {fn:,.0f}")
        print(f"True Negatives (TN):  {tn:,.0f}")
        print("-" * 40)
        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision:     {precision:.4f}")
        print(f"Recall:        {recall:.4f}")
        print(f"F1-Measure:    {f_measure:.4f}")
        print(f"False Pos Rate: {fpr:.4f}")
        print(f"False Neg Rate: {fnr:.4f}")
        print("=" * 40)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f_measure': f_measure,
            'fpr': fpr,
            'fnr': fnr
        }
    except ZeroDivisionError as e:
        print(f"❌ ERROR: Division by zero in metrics calculation: {e}")
        return None

print("✅ Metrics helper function defined")

In [ ]:
# Cell 24: Visualization Helper Function
# =========================================

def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    """
    Plot confusion matrix with consistent styling across all notebooks.

    Args:
        tp, fp, fn, tn: Confusion matrix values
        title: Chart title
    """
    import seaborn as sns
    import matplotlib.pyplot as plt

    matrix_values = [[tp, fn], [fp, tn]]

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        matrix_values,
        annot=True,
        fmt=',.0f',
        cmap='Purples',
        linewidths=2,
        linecolor='white',
        xticklabels=['Predicted Positive', 'Predicted Negative'],
        yticklabels=['Actual Positive', 'Actual Negative'],
        cbar=False,
        annot_kws={"size": 16, "weight": "bold"}
    )

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)
    plt.tick_params(labelsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Visualization helper function defined")

In [ ]:
# Cell 25: Calculate Confusion Matrix & Metrics
# =========================================

total_events = FinalResults.count()

print(f"\nTotal Events Analyzed (N): {total_events}")

# ⚠️ While this is part of the file it HAS NOT been configured or used.
# Logic is flawed.
THRESHOLD = 66

tp = FinalResults.filter(F.col("final_anomaly_score") > THRESHOLD).count()
fp = FinalResults.filter(F.col("final_anomaly_score") <= THRESHOLD).count()
fn = total_events - tp
tn = total_events - tp - fp - fn

print(f"\nConfusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

metrics = calculate_metrics(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    total_events=total_events
)

In [ ]:
# Cell 26: Plot Confusion Matrix
# =========================================

plot_confusion_matrix(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    title='Confusion Matrix - Process Anomaly Detection'
)

## Conclusion

This notebook successfully:
- Configured Spark with environment variables (including GPU support)
- Loaded and filtered Sysmon logs
- Sessionized process events into feature vectors
- Trained Seq2Seq Autoencoder for sequence anomaly detection
- Trained Isolation Forest for statistical anomaly detection
- Calculated rarity scores for context
- Combined scores using weighted ensemble approach
- Generated classification metrics and confusion matrix

**Key Insights:**
- Hybrid approach (Deep Learning + Statistical) provides better detection than either alone
- Rarity boosting helps identify unique/singleton processes which may be threats
- Z-score normalization accounts for process-specific baseline behavior
- Tunable weights allow adjusting sensitivity to data characteristics

**Next Steps:**
- Investigate top anomalous processes by final score
- Review Seq2Seq sequences with high reconstruction error
- Check Isolation Forest outliers for rare patterns
- Tune hyperparameters (seq_len, vocab, n_estimators, contamination, weights)
- Update threshold (PROCESS_FINAL_ANOMALY_THRESHOLD) for desired false positive rate
- Update configuration for your specific environment
- Consider implementing automated alerts for high-score anomalies

**Configuration Changes:**
- Modify `.env` file or set environment variables for:
  - Spark cluster settings (master, GPU flags, memory, cores)
  - MinIO credentials and endpoint
  - Data paths
  - Model hyperparameters:
    - Seq2Seq: seq_len, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers, num_epochs
    - Isolation Forest: n_estimators, contamination, random_state
  - Ensemble weights (lstm_norm, if_norm, rarity_boost)
  - Threshold (PROCESS_FINAL_ANOMALY_THRESHOLD)